In [7]:
import os
from groq import Groq
import requests
from simpleeval import simple_eval


In [8]:
client = Groq(api_key=os.environ['llm_key'])

In [9]:
chat_completion = client.chat.completions.create(messages=[
    {
        "role": "user",
        "content": "Today we are creating an agent with ReACT.",
    }
],
    model = "llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)


ReACT (Reasoning about Autonomous Cognitive Technologies) is a framework for creating autonomous agents. I'd be happy to help you create an agent with ReACT.

To get started, can you tell me a bit more about the agent you want to create? What kind of tasks do you want it to perform, and what environment will it be operating in? For example, will it be interacting with a physical environment, or will it be operating in a virtual world?

Also, what specific aspects of ReACT do you want to focus on? Do you want to create a simple reflex agent, a model-based agent, or something more complex like a planning-based or learning-based agent?

Let me know and I'll do my best to assist you in creating your agent with ReACT!


In [10]:
def get_weather(location: str) -> str:
    weather_data = {
        "Antalya": "Sunny, 28°C",
        "Karlsruhe": "Rainy, 17°C",
        "Helsinki": "Foggy, 6°C"
    }
    return weather_data.get(location, f"No information found for the city of {location}.")


In [11]:
def get_weather_mock(location: str) -> str:
    weather_data = {"Antalya": "Sunny, 28°C", "Karlsruhe": "Rainy, 17°C"}
    return weather_data.get(location, f"No mock data for {location}.")



In [12]:
def get_weather_withAPI(location: str) -> str:
    try:
        #1.step Trying to get the geocoding
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=en&format=json"
        geo_response = requests.get(geo_url).json()

        if "results"  not in geo_response:
            return "no information found for the city of {location}."

        lat = geo_response["results"][0]["latitude"]
        lon = geo_response["results"][0]["longitude"]

        #2.step Wİth coordinates find the weather
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_response = requests.get(weather_url).json()

        current = weather_response["current_weather"]
        temperature = current["temperature"]
        wind_speed = current["windspeed"]


        return f"Current temperature in {location} is {temperature}°C. Wind speed is {wind_speed} km/h."

    except Exception as e:
        # just in case connections is established
        return f"API Error: {str(e)}"









In [13]:
def calculate(answer: str)-> str:
    try:
        answer = answer.strip()
        result = simple_eval(answer)
        return f"Calculated answer: {result}"

    except Exception as e:
        return f"Calculation Error: {str(e)}"



In [14]:
system_prompt = """
You are an AI assistant that thinks logically step-by-step. If you are asked about the weather, use the following tool:

- get_weather: Used to find out the weather of a city. It takes only the city name (e.g., "Istanbul") as a parameter.
-calculate:Used to find out the math problem that has been given such as  "25-20 " as a parameter.

To answer the question asked, strictly use the following format:

Thought: I am thinking step-by-step about what I need to do to answer the question.
Action: [The name of the tool to use. Only use get_weather]
Action Input: [The parameter to send to the tool]

If you used an Action, STOP here. I will provide you with the tool's result as "Observation: ...".

If you know the answer yourself or have reached the answer after an Observation, use the following format:

Thought: I know the answer.
Final Answer: [The final answer to provide to the user]
"""

In [15]:
def parse_llm_output(output_text:str):
    action = None
    action_input = None

    lines = output_text.split("\n")
    for line in lines:
        if line.startswith("Action:"):
            action = line.split("Action:")[1].strip()
        elif line.startswith("Action Input:"):
            action_input = line.split("Action Input:")[1].strip()

    return action,action_input

In [16]:
def run_reAct_agent(user_query:str,tools:dict,max_steps:int = 5 ):
    messages = [


            {"role": "system","content":system_prompt},
            {"role": "user","content":user_query}

    ]

    for step in range(max_steps):
        print(f"\n--- Adım {step + 1} ---")

        response = client.chat.completions.create(
            messages=messages,
            model="llama-3.3-70b-versatile",
            temperature=0.0
        )
        llm_output = response.choices[0].message.content
        print(llm_output)

        # 1 modelin ne düşündüğünü eski mesajlara ekle
        messages.append({"role": "assistant","content":llm_output})

        # 2 final cevaba ulaşma kontrolü
        if "Final Answer:" in llm_output:
            final_answer = llm_output.split("Final Answer:")[1].strip()
            print(final_answer)
            return final_answer


        # 3 Aracı kullnaması lazımsa veri temizleme
        action,action_input = parse_llm_output(llm_output)

        if action:
            if action in tools:
                observation = tools[action](action_input)
                print(f"\n[SİSTEM] Observation: {observation}")
                messages.append({"role": "user","content": f"observation: {observation}"})
            else:
                error_msg = f"There is no tool called{action} ."
                print(f"\n[SİSTEM] Observation: {error_msg}")
                messages.append({"role": "user", "content": f"Observation: {error_msg}"})

    return "Sorry you have reached the limit."







In [17]:

my_tools_real = {"get_weather":get_weather_withAPI,"calculate":calculate}
print("\nSONUÇ:",run_reAct_agent(" what is the weather for Karlsruhe",tools=my_tools_real))


--- Adım 1 ---
Thought: I need to find out the weather for a specific city, so I should use the tool designed for that purpose.
Action: get_weather
Action Input: Karlsruhe

[SİSTEM] Observation: Current temperature in Karlsruhe is 34.7°C. Wind speed is 5.9 km/h.

--- Adım 2 ---
Thought: I have received the weather information for Karlsruhe, which includes the current temperature and wind speed.
Final Answer: The current temperature in Karlsruhe is 34.7°C with a wind speed of 5.9 km/h.
The current temperature in Karlsruhe is 34.7°C with a wind speed of 5.9 km/h.

SONUÇ: The current temperature in Karlsruhe is 34.7°C with a wind speed of 5.9 km/h.


In [19]:
print("\nSONUÇ:",run_reAct_agent(" what is the sqaure root of 16",tools=my_tools_real))


--- Adım 1 ---
Thought: To find the square root of 16, I need to calculate it.
Action: calculate
Action Input: sqrt(16)

[SİSTEM] Observation: Calculation Error: Function 'sqrt' not defined, for expression 'sqrt(16)'.

--- Adım 2 ---
Thought: The calculate function does not support the sqrt function, so I need to find the square root manually or use a different approach. The square root of 16 is a number that, when multiplied by itself, gives 16. I know that 4 * 4 = 16.
Final Answer: 4
4

SONUÇ: 4
